# 종목 하나로 보는 블록 진행

`01_universe_to_signal.ipynb`와 같은 그래프를, 시세만 한 종목으로 줄여서 실행한다.
블록마다 입력이 어디서 오고 출력이 몇 줄인지, 그 줄의 값이 무엇인지 순서대로 본다.

기본 종목은 MTD다. 2026-09-29 종가가 1,000달러를 넘고, RSI 조건과 60일선 조건이 모두 참이다.
`SYMBOL`만 바꾸면 다른 종목으로 같은 경로를 볼 수 있다. 종가가 1,000달러 미만이면 `최소 가격` 출력부터 행이 없다.

In [2]:
import sys
from dataclasses import fields, is_dataclass
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "ai_studio").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from ai_studio.n02_schema import StrategyEdge, StrategyGraph, StrategyNode
from ai_studio.n03_market_data import load_ohlc
from ai_studio.n08_blocks import history_from_bars
from ai_studio.n10_functions import get_function
from ai_studio.n13_connect import validate_graph
from ai_studio.n14_engine import ExecutionContext, execute, execution_order

SYMBOL = "MTD"
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

## 1. 시세를 한 종목만 남긴다

`universe.compose`는 넘겨받은 일봉에 있는 종목을 모두 유니버스로 만든다.
여기서는 그 일봉을 `SYMBOL` 하나만 남긴 뒤 넘긴다.

In [3]:
bars = tuple(bar for bar in load_ohlc() if bar.symbol == SYMBOL)
if not bars:
    raise ValueError(f"{SYMBOL} 일봉이 없습니다.")
history = history_from_bars(bars)
as_of = max(bar.date for bar in bars)
last = history[SYMBOL][-1]

print(f"{SYMBOL}  거래일 {len(bars)}  기간 {bars[0].date} ~ {as_of}")
print(f"기준일 시가 {last.open:,.2f}  고가 {last.high:,.2f}  저가 {last.low:,.2f}  종가 {last.close:,.2f}")
pd.DataFrame(bars[:3])

MTD  거래일 251  기간 2025-09-30 ~ 2026-09-29
기준일 시가 1,540.42  고가 1,552.71  저가 1,523.54  종가 1,537.33


,date,symbol,open,high,low,close,volume
0,2025-09-30,MTD,1190.069946,1231.569946,1190.069946,1227.609985,134600.0
1,2025-10-01,MTD,1227.729980,1292.099976,1227.729980,1289.819946,200500.0
2,2025-10-02,MTD,1278.400024,1302.790039,1261.119995,1300.540039,163000.0


## 2. 그래프

블록 구성은 01 노트북과 같다. 유니버스가 한 종목이라 각 출력도 그 종목만 담는다.
교차 이벤트만 예외다. 그 블록은 기준일 하루가 아니라, 저장된 일봉에서 교차한 날마다 한 줄을 만든다.

In [4]:
nodes = (
    StrategyNode("universe", "universe.compose", {"market": "US_EQUITY", "benchmark": "SPX"}),
    StrategyNode("filter", "universe.exclude", {"minListingDays": 148}),
    StrategyNode("price", "universe.min-price", {"minClose": 1000}),
    StrategyNode("rsi", "indicator.rsi", {"period": 14}),
    StrategyNode("oversold", "condition.threshold", {"operator": "lt", "threshold": 30}),
    StrategyNode("strong", "condition.threshold", {"operator": "gt", "threshold": 50}),
    StrategyNode("trend", "momentum.trend", {"method": "price_above_ma", "slowDays": 60}),
    StrategyNode("entry_rule", "logic.combine", {"operator": "and"}),
    StrategyNode("cross", "momentum.ma-cross", {"fastDays": 20, "slowDays": 60}),
    StrategyNode("state_decision", "strategy.state", {}),
    StrategyNode("event_decision", "strategy.event", {}),
    StrategyNode("hybrid_decision", "strategy.hybrid", {}),
)
edges = (
    StrategyEdge("universe", "universe", "filter", "universe"),
    StrategyEdge("filter", "universe", "price", "universe"),
    StrategyEdge("price", "universe", "rsi", "universe"),
    StrategyEdge("rsi", "value", "oversold", "value"),
    StrategyEdge("rsi", "value", "strong", "value"),
    StrategyEdge("price", "universe", "trend", "universe"),
    StrategyEdge("trend", "condition", "entry_rule", "conditions"),
    StrategyEdge("strong", "condition", "entry_rule", "conditions"),
    StrategyEdge("price", "universe", "cross", "universe"),
    StrategyEdge("price", "universe", "state_decision", "universe"),
    StrategyEdge("entry_rule", "condition", "state_decision", "condition"),
    StrategyEdge("price", "universe", "event_decision", "universe"),
    StrategyEdge("cross", "entry", "event_decision", "entry"),
    StrategyEdge("cross", "exit", "event_decision", "exit"),
    StrategyEdge("price", "universe", "hybrid_decision", "universe"),
    StrategyEdge("entry_rule", "condition", "hybrid_decision", "eligible"),
    StrategyEdge("cross", "entry", "hybrid_decision", "entry"),
    StrategyEdge("cross", "exit", "hybrid_decision", "exit"),
)
graph = StrategyGraph(nodes, edges)
check = validate_graph(graph)
print(f"연결 검사: {check.ok}  {check.reason or ''}")
print("실행 순서:", " → ".join(node.id for node in execution_order(graph)))

연결 검사: True  
실행 순서: universe → filter → price → cross → event_decision → rsi → oversold → strong → trend → entry_rule → hybrid_decision → state_decision


## 3. 한 번 실행

`as_of`는 그 종목 일봉의 마지막 날짜다. `execute`는 이 날짜로 그래프를 한 번 돌리고, 블록 출력을 `result`에 저장한다.

In [5]:
context = ExecutionContext(as_of, history)
result = execute(graph, context)
print(f"기준일 {as_of}  블록 {len(result.outputs)}개")

기준일 2026-09-29  블록 12개


## 4. 블록 순서대로 출력

각 칸은 앞 블록의 출력을 입력으로 받는다. 표는 그 블록이 새로 만든 값이다.

In [6]:
def frame_table(value):
    if hasattr(value, "rows"):
        rows = value.rows
    elif hasattr(value, "signals"):
        rows = value.signals
    else:
        rows = (value,)
    if not rows:
        return pd.DataFrame()
    if is_dataclass(rows[0]):
        return pd.DataFrame([{item.name: getattr(row, item.name) for item in fields(rows[0])} for row in rows])
    return pd.DataFrame(rows)


for node in execution_order(graph):
    spec = get_function(node.function_id)
    incoming = [edge for edge in edges if edge.to_node == node.id]
    print(f"\n{node.id}   {spec.name}")
    if incoming:
        print("  입력 " + ", ".join(f"{edge.from_node}.{edge.from_port} → {edge.to_port}" for edge in incoming))
    else:
        print("  입력 없음. 일봉과 기준일은 실행 컨텍스트에서 읽는다.")
    for port, value in result.outputs[node.id].items():
        label = next(item.label for item in spec.outputs if item.key == port)
        table = frame_table(value)
        print(f"  출력 {label} ({port})  {len(table)}행")
        display(table)


universe   유니버스 구성
  입력 없음. 일봉과 기준일은 실행 컨텍스트에서 읽는다.
  출력 종목 집합 (universe)  1행


,instrument_id,eligible,metadata
0,MTD,True,None



filter   투자 제외 조건
  입력 universe.universe → universe
  출력 종목 집합 (universe)  1행


,instrument_id,eligible,metadata
0,MTD,True,None



price   최소 가격
  입력 filter.universe → universe
  출력 종목 집합 (universe)  1행


...

,instrument_id,matched
0,MTD,True



hybrid_decision   상태·이벤트 혼합 전략
  입력 price.universe → universe, entry_rule.condition → eligible, cross.entry → entry, cross.exit → exit
  출력 전략 판단 (result)  1행


,symbol,eligible,score,action,desired_state,target_weight,reason_codes,event_id
0,MTD,True,None,HOLD,UNCHANGED,None,(),None



state_decision   상태 평가 전략
  입력 price.universe → universe, entry_rule.condition → condition
  출력 전략 판단 (result)  1행


,symbol,eligible,score,action,desired_state,target_weight,reason_codes,event_id
0,MTD,True,None,REBALANCE,LONG,1.0,(),None


## 5. 기준일 한 줄로 모아서 보기

조건과 전략 판단은 기준일 값이다. 교차는 전체 기간의 발생 횟수다.
이벤트 전략의 진입은 기준일에 교차가 있을 때만 나온다. 위의 전체 횟수와는 별개다.

In [7]:
def one(node_id, port, column):
    table = frame_table(result.port(node_id, port))
    if table.empty:
        return None
    return table.iloc[0][column]


entry_rows = frame_table(result.port("cross", "entry"))
exit_rows = frame_table(result.port("cross", "exit"))
state = frame_table(result.port("state_decision", "result"))
event = frame_table(result.port("event_decision", "result"))
hybrid = frame_table(result.port("hybrid_decision", "result"))

summary = pd.DataFrame([{
    "symbol": SYMBOL,
    "as_of": as_of,
    "close": last.close,
    "rsi": one("rsi", "value", "value"),
    "oversold": one("oversold", "condition", "matched"),
    "strong": one("strong", "condition", "matched"),
    "trend": one("trend", "condition", "matched"),
    "entry_rule": one("entry_rule", "condition", "matched"),
    "state_action": None if state.empty else state.iloc[0]["action"],
    "state_weight": None if state.empty else state.iloc[0]["target_weight"],
    "event_action": None if event.empty else event.iloc[0]["action"],
    "hybrid_action": None if hybrid.empty else hybrid.iloc[0]["action"],
    "entry_events": len(entry_rows),
    "exit_events": len(exit_rows),
}])
summary

,symbol,as_of,close,rsi,oversold,strong,trend,entry_rule,state_action,state_weight,event_action,hybrid_action,entry_events,exit_events
0,MTD,2026-09-29,1537.329956,85.363246,False,True,True,True,REBALANCE,1.0,HOLD,HOLD,3,3


In [8]:
events = pd.concat([entry_rows, exit_rows], ignore_index=True)
if events.empty:
    print("교차한 날이 없다.")
else:
    events["date"] = events["as_of"].dt.date.astype(str)
    events[["instrument_id", "date", "kind"]].sort_values(["date", "kind"])

## 6. 1년 백테스트 — 상태 전략

기준일을 거래일마다 바꿔 같은 그래프를 다시 실행한다. 판단은 `state_decision`이다.
조건이 참이면 목표 비중을 다시 적고, 거짓이면 비중 0이다. 신호는 그날 종가, 체결은 다음 거래일 시가다.

상장일수 148일 조건 때문에 첫 평가일은 2026-05-01이다. 그 전에는 유니버스에서 빠진다.
전략이 낸 비중은 참일 때 1이다. 주문은 종가 기준으로 수량을 정하고 다음 시가에 사므로,
시가가 종가보다 높으면 현금이 부족해진다. 모멘텀 백테스트와 같이 현금 20%를 남기고 80%만 주문한다.
체결마다 수수료 `FEE_RATE`와 슬리피지 `SLIPPAGE_RATE`가 빠진다. 매수는 시가보다 비싸게, 매도는 시가보다 싸게 체결된다.
이 셀은 백테스트 요약과 함께 매매 내역 CSV를 `outputs/{종목}_trades.csv`에 쓴다. 매수금과 매도금은 수수료를 넣기 전 체결대금이다.

In [13]:
from dataclasses import replace
from datetime import datetime, time, timezone

from pathlib import Path

from ai_studio.n03_market_data import DATA_PATH
from ai_studio.n07_runtime import MarketBar
from ai_studio.n09_backtest import run_backtest, write_trades_csv

CLOSE = time(21, 0, tzinfo=timezone.utc)
OPEN = time(14, 30, tzinfo=timezone.utc)
CASH_WEIGHT = 0.2
INITIAL_CASH = 1_000_000
FEE_RATE = 0.05
SLIPPAGE_RATE = 0.05

dates = [bar.date for bar in bars]
by_date = {bar.date: bar for bar in bars}
results = []
reference_prices = {}
next_bars = {}
decisions = []

for index in range(147, len(dates) - 1):
    day = dates[index]
    nxt = dates[index + 1]
    decision = execute(graph, ExecutionContext(day, history)).port("state_decision", "result")
    invested = 1 - CASH_WEIGHT
    signals = tuple(
        replace(row, target_weight=None if row.target_weight is None else row.target_weight * invested)
        for row in decision.signals
    )
    decision = replace(
        decision,
        signals=signals,
        target_weights=tuple((row.symbol, row.target_weight or 0.0) for row in signals),
    )
    results.append(decision)
    reference_prices[decision.as_of] = {SYMBOL: by_date[day].close}
    next_stamp = datetime.combine(datetime.strptime(nxt, "%Y-%m-%d").date(), CLOSE)
    next_bars[decision.as_of] = {
        SYMBOL: MarketBar(
            next_stamp, next_stamp, SYMBOL,
            by_date[nxt].open, by_date[nxt].close,
            open_at=datetime.combine(next_stamp.date(), OPEN),
        )
    }
    weight = signals[0].target_weight if signals else 0.0
    decisions.append({"as_of": day, "close": by_date[day].close, "target_weight": weight or 0.0})

backtest = run_backtest(
    results, reference_prices, next_bars,
    initial_cash=INITIAL_CASH, fee_rate=FEE_RATE, slippage_rate=SLIPPAGE_RATE,
)
trade_path = write_trades_csv(
    backtest,
    Path(DATA_PATH).parents[1] / "outputs" / f"{SYMBOL}_trades.csv",
    initial_cash=INITIAL_CASH,
    ohlc={(bar.symbol, bar.date): bar for bar in bars},
)
curve = pd.DataFrame(backtest.portfolio_history)
last = backtest.portfolio_history[-1]
first_fill = dates[148]
shares = int(INITIAL_CASH * (1 - CASH_WEIGHT) // by_date[first_fill].open)
held_to_end = shares * by_date[dates[-1]].close + (INITIAL_CASH - shares * by_date[first_fill].open)

print(f"매매 내역 {trade_path}  {len(backtest.trade_log)}건")
print(f"평가 {len(curve)}회  보유로 표시된 날 {sum(row['target_weight'] > 0 for row in decisions)}일")
print(f"체결 {len(backtest.trade_log)}건  수수료 {FEE_RATE:.2%}  슬리피지 {SLIPPAGE_RATE:.2%}")
print(
    f"수수료 합계 {sum(fill.fee for fill in backtest.trade_log):,.0f}  "
    f"슬리피지 합계 {sum(fill.slippage for fill in backtest.trade_log):,.0f}"
)
print(f"최종 평가액 {last.total_value:,.0f}  누적 수익률 {last.cumulative_return:.2%}")
print(f"최대 낙폭 {min(row.drawdown for row in backtest.portfolio_history):.2%}")
print(f"같은 기간 같은 현금 비중으로 보유만 했을 때 {held_to_end:,.0f}  ({held_to_end / INITIAL_CASH - 1:.2%})")

previous = None
print("\n보유 전환")
for row in decisions:
    held = row["target_weight"] > 0
    if held != previous:
        label = "진입" if held else ("현금 시작" if previous is None else "청산")
        print(f"  {label}  {row['as_of']}  종가 {row['close']:,.2f}")
        previous = held

for row in backtest.portfolio_history:
    if row.as_of.day in (1, 15) or row is backtest.portfolio_history[0] or row is backtest.portfolio_history[-1]:
        bar = "█" * max(0, round(row.cumulative_return * 40))
        print(f"{row.as_of.date()}  {row.cumulative_return:+7.2%}  {row.total_value:12,.0f}  {bar}")

평가 103회  보유로 표시된 날 46일
체결 39건  수수료 5.00%  슬리피지 5.00%
수수료 합계 263,680  슬리피지 합계 262,734
최종 평가액 470,454  누적 수익률 -52.95%
최대 낙폭 -54.00%
같은 기간 같은 현금 비중으로 보유만 했을 때 1,176,447  (17.64%)

보유 전환
  현금 시작  2026-05-01  종가 1,267.07
  진입  2026-05-06  종가 1,327.04
  청산  2026-05-07  종가 1,319.29
  진입  2026-06-25  종가 1,243.42
  청산  2026-07-22  종가 1,285.90
  진입  2026-07-23  종가 1,331.87
  청산  2026-08-20  종가 1,393.23
  진입  2026-09-17  종가 1,410.44
  청산  2026-09-18  종가 1,392.32
  진입  2026-09-21  종가 1,430.44
2026-05-04   +0.00%     1,000,000  
2026-05-15  -23.71%       762,867  
2026-06-01  -23.71%       762,867  
2026-06-15  -23.71%       762,867  
2026-07-01  -27.73%       722,676  
2026-07-15  -27.15%       728,453  
2026-09-01  -41.95%       580,533  
2026-09-15  -41.95%       580,533  
2026-09-29  -52.95%       470,454  


2026-05-01부터 마지막 거래일까지 하루씩 다시 계산한다.
`oversold`, `strong`, `trend`, `entry_rule`은 그날의 참·거짓이다.
`entry`는 전날 거짓이던 `entry_rule`이 참으로 바뀐 날이고, `exit`는 참에서 거짓으로 바뀐 날이다.
6절의 매수·매도 신호일과 같다. 체결은 그 다음 거래일 시가다.
`open` `high` `low` `close`는 그날 OHLC다.
`shares`는 그날 마감 보유 수량이다. 주식이 없으면 0이다.
`buy_qty` `sell_qty`는 그날 시가에 실제 사고 판 수량이다. 거래가 없으면 0이다.
`buy_amount`는 수수료를 더하기 전 매수 체결대금이다. 수량 × 시가 × (1+슬리피지율).
`sell_amount`는 수수료를 빼기 전 매도 체결대금이다. 수량 × 시가 × (1−슬리피지율).
현금은 매수일에 `buy_amount + fee`만큼 줄고, 매도일에 `sell_amount − fee`만큼 늘어난다.
`slippage`는 시가와 체결가 차이에 수량을 곱한 금액이고, `fee`는 체결가에 수수료율을 곱한 금액이다.
`stock_value`는 마감 보유 수량 × 그날 종가다. `portfolio_value`는 현금과 이 평가금액을 더한 값이다.
신호는 전날 종가에서 나오고, 수량도 그 종가로 정한다. 체결 금액은 다음 행의 시가부터 바뀐다.

In [10]:
dates = [bar.date for bar in bars]
by_date = {bar.date: bar for bar in bars}
rows = []

for day in dates[dates.index("2026-05-01"):]:
    run = execute(graph, ExecutionContext(day, history))

    def flag(node_id):
        matched = run.port(node_id, "condition").rows
        return bool(matched[0].matched) if matched else None

    rsi_rows = run.port("rsi", "value").rows
    rows.append({
        "as_of": day,
        "open": by_date[day].open,
        "high": by_date[day].high,
        "low": by_date[day].low,
        "close": by_date[day].close,
        "rsi": None if not rsi_rows else round(rsi_rows[0].value, 2),
        "oversold": flag("oversold"),
        "strong": flag("strong"),
        "trend": flag("trend"),
        "entry_rule": flag("entry_rule"),
    })

previous = False
for row in rows:
    held = bool(row["entry_rule"])
    row["entry"] = held and not previous
    row["exit"] = (not held) and previous
    previous = held

ledger = {row.as_of.date().isoformat(): row for row in backtest.portfolio_history}
shares_of = {
    row.as_of.date().isoformat(): row.quantity
    for row in backtest.position_history
    if row.symbol == SYMBOL and row.quantity
}
cash = INITIAL_CASH
stock_value = 0.0
for row in rows:
    book = ledger.get(row["as_of"])
    if book is not None:
        cash = book.cash
        stock_value = book.securities_value
        shares = shares_of.get(row["as_of"], 0.0)
    else:
        shares = 0.0
        stock_value = 0.0
    portfolio_value = cash + stock_value
    row["shares"] = shares
    row["cash"] = round(cash, 2)
    row["stock_value"] = round(stock_value, 2)
    row["portfolio_value"] = round(portfolio_value, 2)
    row["cash_weight"] = round(cash / portfolio_value, 4) if portfolio_value else None

from ai_studio.n07_runtime import trade_notional

costs = {}
for fill in backtest.trade_log:
    day = fill.filled_at.date().isoformat()
    slot = costs.setdefault(day, {
        "fee": 0.0, "slippage": 0.0,
        "buy_qty": 0.0, "sell_qty": 0.0,
        "buy_amount": 0.0, "sell_amount": 0.0,
    })
    buy_amount, sell_amount = trade_notional(fill)
    slot["fee"] += fill.fee
    slot["slippage"] += fill.slippage
    slot["buy_qty"] += fill.quantity if fill.side == "BUY" else 0.0
    slot["sell_qty"] += fill.quantity if fill.side == "SELL" else 0.0
    slot["buy_amount"] += buy_amount
    slot["sell_amount"] += sell_amount
empty_cost = {"fee": 0.0, "slippage": 0.0, "buy_qty": 0.0, "sell_qty": 0.0, "buy_amount": 0.0, "sell_amount": 0.0}
for row in rows:
    slot = costs.get(row["as_of"], empty_cost)
    row["fee"] = round(slot["fee"], 2)
    row["slippage"] = round(slot["slippage"], 2)
    row["buy_qty"] = slot["buy_qty"]
    row["sell_qty"] = slot["sell_qty"]
    row["buy_amount"] = round(slot["buy_amount"], 2)
    row["sell_amount"] = round(slot["sell_amount"], 2)

daily = pd.DataFrame(rows)
print(f"{daily['as_of'].iloc[0]} ~ {daily['as_of'].iloc[-1]}  {len(daily)}일")
print(f"진입 {int(daily['entry'].sum())}회  청산 {int(daily['exit'].sum())}회")

from IPython.display import HTML, display

shown = daily.copy()
for column in ("open", "high", "low", "close"):
    shown[column] = shown[column].map(lambda value: f"{value:,.2f}")
shown["shares"] = shown["shares"].map(lambda value: f"{value:,.0f}")
shown["buy_qty"] = shown["buy_qty"].map(lambda value: f"{value:,.0f}")
shown["sell_qty"] = shown["sell_qty"].map(lambda value: f"{value:,.0f}")
for column in ("buy_amount", "sell_amount", "fee", "slippage"):
    shown[column] = shown[column].map(lambda value: f"{value:,.2f}")
shown["cash"] = shown["cash"].map(lambda value: f"{value:,.0f}")
shown["stock_value"] = shown["stock_value"].map(lambda value: f"{value:,.0f}")
shown["portfolio_value"] = shown["portfolio_value"].map(lambda value: f"{value:,.0f}")
shown["cash_weight"] = shown["cash_weight"].map(lambda value: f"{value:.1%}")
shown = shown[
    ["as_of", "open", "high", "low", "close",
     "rsi", "oversold", "strong", "trend", "entry_rule", "entry", "exit",
     "shares", "buy_qty", "buy_amount", "sell_qty", "sell_amount",
     "slippage", "fee", "cash", "stock_value", "portfolio_value", "cash_weight"]
]
display(HTML(
    '<div style="max-height:640px; max-width:100%; overflow:auto">'
    + shown.to_html(index=False)
    + "</div>"
))

2026-05-01 ~ 2026-09-29  104일
진입 5회  청산 4회


as_of,open,close,rsi,oversold,strong,trend,entry_rule,entry,exit,shares,fee,slippage,cash,stock_value,portfolio_value,cash_weight
2026-05-01,"1,281.83","1,267.07",36.29,False,False,False,False,False,False,0,0.00,0.00,"1,000,000",0,"1,000,000",100.0%
2026-05-04,"1,259.46","1,260.54",36.02,False,False,False,False,False,False,0,0.00,0.00,"1,000,000",0,"1,000,000",100.0%
2026-05-05,"1,282.75","1,302.42",47.80,False,False,True,False,False,False,0,0.00,0.00,"1,000,000",0,"1,000,000",100.0%
2026-05-06,"1,326.49","1,327.04",55.93,False,True,True,True,True,False,0,0.00,0.00,"1,000,000",0,"1,000,000",100.0%
2026-05-07,"1,333.91","1,319.29",45.96,False,False,True,False,False,True,602,803.82,803.01,"195,379","794,213","989,592",19.7%
2026-05-08,"1,193.05","1,124.46",27.37,True,False,False,False,False,False,0,717.50,718.22,"912,160",0,"912,160",100.0%
2026-05-11,"1,116.66","1,078.53",24.81,True,False,False,False,False,False,0,0.00,0.00,"912,160",0,"912,160",100.0%
2026-05-12,"1,086.22","1,072.45",25.23,True,False,False,False,False,False,0,0.00,0.00,"912,160",0,"912,160",100.0%
2026-05-13,"1,056.35","1,025.53",25.17,True,False,False,False,False,False,0,0.00,0.00,"912,160",0,"912,160",100.0%
2026-05-14,"1,030.20","1,041.91",24.90,True,False,False,False,False,False,0,0.00,0.00,"912,160",0,"912,160",100.0%
